# Quantum random-bit circuit


Material development: **Gage Plott**.

Run this worked example with the video; pause whenever you need time. Nothing is
graded or submitted. No earlier notebook is needed. Start with the
[quickstart](../../intro/START_HERE.md) if the environment is not ready.

## 1. Check the notebook environment

Click a code cell, press **Shift+Enter**, and wait for its running indicator to
finish. Run from top to bottom. A **kernel** is the running Python session: it
remembers names and values until restarted. Choose **Python (PHYS 3810 Intro)**,
then compare the `Interpreter:` path below with the path printed in your activated
setup terminal. A matching display name alone is not enough.

Run this support cell unchanged. `from ... import ...` loads tools we will use.
The first line tells Jupyter to display figures here. The checks below are
software checks, not questions you must answer. If a check fails, return to the
quickstart's recovery steps.

In [ ]:
%matplotlib inline
import struct
import sys
from importlib.metadata import version

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit import ClassicalRegister, QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector
from qiskit.visualization import plot_histogram

assert sys.implementation.name == "cpython" and sys.version_info[:2] == (3, 12)
assert struct.calcsize("P") * 8 == 64, "Select 64-bit Python 3.12."
assert version("qiskit") == "2.6.1", "Select the pinned course environment."
assert version("qiskit-aer") == "0.18.2", "Select the pinned course environment."
print("Python:", sys.version.split()[0])
print("Qiskit:", version("qiskit"), "| Aer:", version("qiskit-aer"))
print("Interpreter:", sys.executable)

## 2. Choose the shot count

`=` stores a value under a name. Here `shots` is a whole number. A line beginning
with `#` is a comment for the reader. `print(...)` displays a result.

A **shot** means preparing the circuit again and measuring once. Keep the defaults
for the first run. The simulator seed makes the example reproducible in this
environment: a complete rerun can repeat the same samples. Exact sequences are
not promised across software versions.

In [ ]:
shots = 128  # Change only this number in the follow-along edit.
seed = 2026  # Keep this fixed for the demonstration.
assert isinstance(shots, int) and not isinstance(shots, bool) and shots > 0
print(f"Requested shots: {shots}; simulator seed: {seed}")

## 3. Prepare the RNG circuit

`QuantumCircuit(1)` makes one qubit in state $|0\rangle$. Python counts from zero,
so `rng.h(0)` applies the **Hadamard (H)** gate to qubit 0. The dot calls an
operation on that circuit. H prepares
$|+\rangle=(|0\rangle+|1\rangle)/\sqrt{2}$, with exact probabilities $1/2$ and $1/2$.

This is a **quantum random-bit circuit**, simulated by classical software on this
laptop. The bits below are **not physical quantum entropy or cryptographic
randomness**. A real quantum device is not contacted.

The first two lines are the circuit preparation. Every execution starts a fresh
circuit, so running the cell again does not append another H. The remaining
lines calculate the ideal probabilities and draw the circuit.

In [ ]:
rng = QuantumCircuit(1)
rng.h(0)

rng_state = Statevector.from_instruction(rng)
rng_probabilities = rng_state.probabilities()
np.testing.assert_allclose(rng_probabilities, [0.5, 0.5], rtol=0, atol=1e-12)
print("Exact probabilities for [0, 1]:", rng_probabilities)
rng_diagram = rng.draw("mpl", scale=1.2)
display(rng_diagram)
plt.close(rng_diagram)

## 4. Sample the circuit

This cell makes a separate measured copy. It maps qubit 0 to classical bit 0 in
the register named `meas`; the preparation `rng` stays unmeasured. The V2
`StatevectorSampler` samples the circuit's predicted probabilities locally.

`get_bitstrings()` returns the actual samples in shot order. The short displayed
sequence comes from those samples, not from rearranging a histogram. A **count**
is how many times a label occurred. The counts add up to `shots`; a finite sample
need not split exactly in half, and a very small sample may contain only one label.

In [ ]:
rng_measured = rng.copy()
rng_measured.add_register(ClassicalRegister(1, "meas"))
rng_measured.measure([0], [0])
rng_result = StatevectorSampler(seed=seed).run([rng_measured], shots=shots).result()[0]
rng_bits = rng_result.data.meas.get_bitstrings()
rng_counts = rng_result.data.meas.get_counts()
assert len(rng_bits) == sum(rng_counts.values()) == shots
assert set(rng_counts) <= {"0", "1"}
assert rng.num_clbits == 0
rng_caption = f"RNG: {shots} simulated shots; seed={seed}; exact P(0)=P(1)=1/2."
print(rng_caption)
print(f"First {min(24, shots)} bits in shot order:", " ".join(rng_bits[:24]))
print("Counts:", rng_counts)

## 5. Plot the counts

The horizontal labels are measured bit values; each bar's height is a count.
Compare the finite sample with the exact probabilities from section 3. More
shots give more observations, not a guarantee of exact balance.

In [ ]:
assert sum(rng_counts.values()) == shots, "Run section 4 after changing shots."
rng_title = f"Quantum random-bit circuit | {shots} simulated shots | seed {seed}"
rng_histogram = plot_histogram(rng_counts, title=rng_title)
display(rng_histogram)
plt.close(rng_histogram)
print(rng_caption)

## 6. Change one number, then run it

1. In **section 2**, change `shots = 128` to `shots = 1024`.
2. **Run the edited section 2 parameter cell with Shift+Enter.** Editing its text
   alone does not change the kernel's remembered value.
3. Run the **section 4 sampling cell**, then the **section 5 plotting cell**.
   Check that the requested total, caption, and plot title all say 1024.
4. Change section 2 back to `shots = 128`, **run section 2**, then **section 4**
   and **section 5** again. Leave `seed = 2026` unchanged.

Section 3's circuit does not depend on the shot count and does not need rebuilding
for this edit. A passed assertion can still describe an old value if section 2
was never executed. If confused, save your working copy and use
**Kernel > Restart Kernel and Run All Cells**; this clears memory and reruns the
current written values in order. A saved picture is only a record of an earlier run.

Next open [Bell-state preparation](02_bell_state.ipynb).

Optional reference: IBM's [StatevectorSampler documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.primitives.StatevectorSampler).